In [2]:
!cd /workspace/my_data

In [2]:
!ls

'convert_to_nemo(1)(1).ipynb'   test.txt


In [3]:
from huggingface_hub import snapshot_download

print("Downloading Qwen weights...")
snapshot_download(
    repo_id="Qwen/Qwen2.5-14B-Instruct", 
    local_dir="./qwen_hf_weights",
    ignore_patterns=["*.gguf", "*.h5"] 
)
print("Download complete!")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 18 files: 100%|██████████| 18/18 [04:16<00:00, 14.27s/it]

Download complete!


In [3]:
!ls /opt/NeMo/scripts/checkpoint_converters/ | grep qwen

convert_qwen2_hf_to_nemo.py
convert_qwen2_nemo_to_hf.py


In [4]:
%pip install transformers==4.44.2

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.23a0+6627725-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.12.0.dev0-py3.12.egg is de

In [1]:
!python /opt/NeMo/scripts/checkpoint_converters/convert_qwen2_hf_to_nemo.py \
    --input_name_or_path "./qwen_hf_weights" \
    --output_path "qwen2.5-14b-instruct.nemo" \
    --precision bf16

[NeMo W 2026-04-22 23:36:41 nemo_logging:405] /opt/megatron-lm/megatron/core/transformer/cuda_graphs.py:741: SyntaxWarning: assertion is always true, perhaps remove parentheses?
      assert (
    
[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.

In [1]:
import json
from transformers import AutoTokenizer

# Load the Qwen tokenizer to handle the special chat formatting
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-14B-Instruct")

nemo_dataset = []

# Open your Hugging Face formatted dataset
with open("instruction_training_data.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)
        
        # Grab the system and user messages
        prompt_messages = [
            {"role": "system", "content": data['messages'][0]['content']},
            {"role": "user", "content": data['messages'][1]['content']}
        ]
        
        # 1. Use the tokenizer to bake in the Qwen specific <|im_start|> tags
        # add_generation_prompt=True adds the final <|im_start|>assistant tag
        nemo_input = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
        
        # 2. Extract the actual answer
        nemo_output = data['messages'][2]['content']
        
        # 3. Save it in the strict format NeMo requires
        nemo_dataset.append({
            "input": nemo_input,
            "output": nemo_output
        })

# Save the new NeMo-ready dataset
with open("nemo_sft_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in nemo_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Successfully converted {len(nemo_dataset)} rows for NeMo!")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully converted 3000 rows for NeMo!


In [7]:
import random

# 1. Load the fully formatted NeMo dataset we created earlier
input_file = "nemo_sft_dataset.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    data_lines = f.readlines()

# 2. Shuffle the data
# We set a seed (42) so if you run this cell again, it splits the exact same way
random.seed(42)
random.shuffle(data_lines)

# 3. Calculate the 90% cutoff index
split_index = int(len(data_lines) * 0.9)

train_lines = data_lines[:split_index]
val_lines = data_lines[split_index:]

# 4. Save the Training Set (90%)
with open("nemo_sft_train.jsonl", "w", encoding="utf-8") as f:
    f.writelines(train_lines)

# 5. Save the Validation Set (10%)
with open("nemo_sft_val.jsonl", "w", encoding="utf-8") as f:
    f.writelines(val_lines)

print(f"Total articles: {len(data_lines)}")
print(f"Saved {len(train_lines)} articles to nemo_sft_train.jsonl")
print(f"Saved {len(val_lines)} articles to nemo_sft_val.jsonl")

Total articles: 3000
Saved 2700 articles to nemo_sft_train.jsonl
Saved 300 articles to nemo_sft_val.jsonl


In [3]:
!python /opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_finetuning.py \
    trainer.max_epochs=3 \
    trainer.precision=bf16 \
    trainer.devices=2 \
    trainer.accelerator="gpu" \
    trainer.log_every_n_steps=1 \
    model.restore_from_path="qwen2.5-14b-instruct.nemo" \
    model.peft.peft_scheme="lora" \
    model.peft.lora_tuning.adapter_dim=32 \
    model.peft.lora_tuning.alpha=64 \
    model.micro_batch_size=4 \
    model.global_batch_size=16 \
    model.activations_checkpoint_granularity="selective" \
    model.activations_checkpoint_method="uniform" \
    model.optim.name="adamw" \
    model.optim.lr=2e-4 \
    model.data.train_ds.file_names=["nemo_sft_train.jsonl"] \
    model.data.train_ds.max_seq_length=2048 \
    model.data.train_ds.concat_sampling_probabilities=[1.0] \
    model.data.train_ds.num_workers=8 \
    model.data.validation_ds.file_names=["nemo_sft_val.jsonl"] \
    model.data.validation_ds.num_workers=8 \
    exp_manager.name="qwen-nemo-lora-2gpu-optimized" \
    exp_manager.create_checkpoint_callback=True

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/triton/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. 

In [4]:
!ls /opt/NeMo/examples/nlp/language_modeling/tuning/ | grep megatron

megatron_gpt_finetuning.py
megatron_gpt_generate.py
megatron_gpt_qat.py
megatron_mamba_finetuning.py
megatron_mamba_generate.py
megatron_t5_finetuning.py
megatron_t5_generate.py


In [1]:
import json

system_prompt = """
You are an expert political data scientist analyzing Greek digital media. Read the provided Greek news article and extract the political ideological leaning of the text on a continuous numerical scale from 0.0 to 1.0.

DEFINITIONS & EXPLICIT ANCHORS:
- 0.00 to 0.15: Far-Left
- 0.16 to 0.35: Left
- 0.36 to 0.45: Center-Left
- 0.46 to 0.55: Center (or strictly Neutral/Objective reporting)
- 0.56 to 0.65: Center-Right
- 0.66 to 0.85: Right
- 0.86 to 1.00: Far-Right

INSTRUCTIONS:
Assign a precise decimal value based on the severity of the bias. For example, a moderately right-wing article might be 0.72, while an extreme left-wing article might be 0.05.

STRICT OUTPUT RULES:
You must respond ONLY with a valid, parsable JSON object. Do NOT include markdown formatting, code blocks, explanations, or any trailing text. 

EXAMPLE OUTPUT:
{"bias": 0.72}
"""
test_queries = []
with open("2sample.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        title = item.get("title", "")
        text = item.get("text", "")
        article_content = f"ΤΙΤΛΟΣ: {title}\nΚΕΙΜΕΝΟ: {text}"
        test_queries.append(article_content)

def format_qwen_prompt(user_query):
    # This matches the ChatML structure Qwen expects
    return f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"

with open("2test_queries_structured.jsonl", "w", encoding="utf-8") as f:
    for query in test_queries:
        structured_input = format_qwen_prompt(query)
        # We save it as 'input' so NeMo knows where to look
        f.write(json.dumps({"input": structured_input}) + "\n")

print("Structured test file created!")

Structured test file created!


In [6]:
#INFEREnCE
!python /opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_generate.py \
    model.restore_from_path="/workspace/my_data/qwen2.5-14b-instruct.nemo" \
    model.peft.restore_from_path="/workspace/my_data/nemo_experiments/qwen-nemo-lora-2gpu-optimized/checkpoints/qwen-nemo-lora-2gpu-optimized.nemo" \
    trainer.devices=1 \
    trainer.precision=bf16 \
    inference.greedy=False \
    inference.temperature=0.1 \
    model.data.test_ds.file_names=["/workspace/my_data/2test_queries_structured.jsonl"] \
    model.data.test_ds.names=["2test"] \
    model.data.test_ds.global_batch_size=1 \
    model.data.test_ds.micro_batch_size=1 \
    model.data.test_ds.write_predictions_to_file=True \
    model.data.test_ds.output_file_path_prefix="/workspace/my_data/2predictions"

/opt/NeMo/examples/nlp/language_modeling/tuning/megatron_gpt_generate.py:44: SyntaxWarning: invalid escape sequence '\['
  """
[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:163: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/selective_scan_interface.py:239: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd

[WARNING  | py.warnings        ]: /usr/local/lib/python3.12/dist-packages/mamba_ssm/ops/triton/layer_norm.py:985: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd

[WARNING  | py.warnings        ]: /usr/local/lib/python

In [17]:
%cd /workspace/my_data/nemo_experiments/qwen-nemo-lora-2gpu-optimized/checkpoints/

# 'c' = create, 'z' = compress (gzip), 'v' = verbose, 'f' = filename
!tar -czvf /workspace/my_data/qwen_greek_lora_final.tar.gz qwen-nemo-lora-2gpu-optimized.nemo

/workspace/my_data/nemo_experiments/qwen-nemo-lora-2gpu-optimized/checkpoints
/bin/bash: line 1: zip: command not found


/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
#COMPARE DISTILLED ACCURARCY
import json
import argparse
import sys
from statistics import mean

def load_data(path):
    """Loads data from either a JSON (list) or JSONL file."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            if content.startswith('['):
                return json.loads(content)
            else:
                # Assume JSONL
                return [json.loads(line) for line in content.split('\n') if line.strip()]
    except Exception as e:
        print(f"Error loading {path}: {e}")
        sys.exit(1)

def extract_biases(data):
    """Maps URL to bias score, handling potential nesting or stringified JSON."""
    biases = {}
    for key, entry in enumerate(data):   
        labels = entry.get('pred', {})
        # Handle case where ai_labels might be a stringified JSON (from legacy formats)
        if isinstance(labels, str):
            try:
                labels = json.loads(labels)
            except:
                continue
        
        bias = labels.get('bias')

        if bias is not None:
            try:
                biases[key] = float(bias)
            except ValueError:
                continue
    return biases

def calculate_mae(biases1, biases2):
    """Calculates Mean Absolute Error between two bias maps."""
    common_keys = set(biases1.keys()) & set(biases2.keys())
    
    if not common_keys:
        return None, 0
        
    errors = []
    for key in common_keys:
        err = abs(biases1[key] - biases2[key])
        errors.append(err)
    
    return mean(errors), len(common_keys)

def main():
    file1 = "2predictions_test_2test_inputs_preds_labels.jsonl"
    file2 = "2sample.jsonl"

    print(f"Loading Source 1: {file1}...")
    data1 = load_data(file1)
    print(f"Loading Source 2: {file2}...")
    data2 = load_data(file2)

    biases1 = extract_biases(data1)
    biases2 = extract_biases(data2)

    mae, count = calculate_mae(biases1, biases2)

    print("\n" + "="*40)
    print("🏆 GROUND TRUTH VALIDATION DASHBOARD 🏆")
    print("="*40)
    print(f"Source A: {file1} ({len(biases1)} samples)")
    print(f"Source B: {file2} ({len(biases2)} samples)")
    print("-" * 40)
    
    if mae is None:
        print("❌ ERROR: No matching articles found between sources.")
        print("Ensure 'url' or 'title' fields match exactly.")
    else:
        print(f"Matched Articles: {count}")
        print(f"Mean Absolute Error (MAE): {mae:.4f}")
        print("-" * 40)
        
        # Interpret result based on 0-1 scale
        if mae < 0.05:
            print("STATUS: OUTSTANDING. High consensus between sources.")
        elif mae < 0.12:
            print("STATUS: GOOD. Minor deviations in bias scoring.")
        elif mae < 0.20:
            print("STATUS: ACCEPTABLE. Broad alignment, but lacks precision.")
        else:
            print("STATUS: DISCREPANCY. Significant difference in bias perception.")
    
    print("="*40 + "\n")

if __name__ == "__main__":
    main()


Loading Source 1: 2predictions_test_2test_inputs_preds_labels.jsonl...
Loading Source 2: 2sample.jsonl...

🏆 GROUND TRUTH VALIDATION DASHBOARD 🏆
Source A: 2predictions_test_2test_inputs_preds_labels.jsonl (20 samples)
Source B: 2sample.jsonl (20 samples)
----------------------------------------
Matched Articles: 20
Mean Absolute Error (MAE): 0.0775
----------------------------------------
STATUS: GOOD. Minor deviations in bias scoring.

